# 04 · Retrieval — Milvus + nv-embedqa + nv-rerankqa

**Purpose** — build the semantic index and show reranking earning its place.

**Prerequisites** — 01 has been run (the database exists) and 00 has been run
(Milvus is up, the embed and rerank NIMs are healthy, and `NIM_MODE=local`).

> The store is **Milvus**. It was LanceDB until pass 26 and the heading here
> still said so; `app/retrieval/backend.py` keeps the LanceDB backend only so
> that migration reverses.

### Scope
Most manager questions ("what is the state of RO-x", "which are blocked",
"what did EMP014 do") are **structured** queries answered by the deterministic
tools — not by retrieval. This serves the narrative slice: *has anyone seen this
fault before*, *what did the last technician say*.

Building the index is a one-off batch job; only the query embedding is per-call.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))          # notebooks/ -> repo root
os.chdir(os.path.abspath(".."))
from dotenv import load_dotenv; load_dotenv()
print("repo root:", os.getcwd())

## Config — check what you are about to build, and where

Read this before building. Two things go wrong silently and both are visible
here: the **store** must be the server on :19530 and not an embedded file, and
the **embedder** must be the one you intend — its width is baked into the
collection, so a mismatch makes every later search fail on a dimension error.

`scripts/build_index.py --check` is the same check from a shell.

In [ ]:
from app.nim.client import resolve
from app.retrieval.backend import backend
from app.retrieval.index import index_meta

base, model, mode = resolve("embed")          # (base_url, model_id, mode)
b = backend()
print(f"  store     {b.uri}   ({b.mode})")
print(f"  embedder  {model}   ({mode})")
if b.mode == "embedded":
    print("  !! that is an embedded Milvus Lite FILE, not the server the API reads.")
    print("     ASOIA_MILVUS_URI is not set in this kernel.")
if mode != "local":
    print("  !! not using the local NIM. On this box that is probably wrong -")
    print("     see notebook 00, and note the hosted embedder is a different width.")
have = index_meta()
print("  on disk  ", have or "nothing recorded yet")

## Execute — build the index (one pass over every update)

In [ ]:
from app.retrieval.index import build
stats = build()
print(stats)
print("recorded:", index_meta())      # dim + embed_model, what the guard reads

## What you should see — search, then rerank

In [ ]:
from app.retrieval.index import search
Q = "grinding noise from the front brakes under braking"
hits = search(Q, k=8, rerank_to=4)
for i, h in enumerate(hits, 1):
    print(f"{i}. {h['ro_number']}  {h.get('vehicle')}  [{h.get('rerank_score', h.get('vector_score')):.3f}]")
    print(f"   {h['text'][:150]}")

## Reranking changes the order

Vector search retrieves broadly; the reranker reads the query against each
passage properly. Compare the top 4 with and without it.

In [ ]:
raw = search(Q, k=6, rerank_to=None)
rer = search(Q, k=6, rerank_to=4)
print("vector only :", [h["ro_number"] for h in raw[:4]])
print("reranked    :", [h["ro_number"] for h in rer])

## Agent-tool shape — passages with citations

In [ ]:
from app.retrieval.index import search_updates
r = search_updates("battery keeps going flat overnight", k=3)
print("citations:", r["citations"])
for p in r["passages"]:
    print(f"  [{p['update_id']}] {p['ro_number']} {p['by']}: {p['text'][:110]}")